In [14]:
# Ячейка 1. Проверка данных Inside Airbnb, Стамбул, выпуск 2026-09-25
!pip -q install langdetect scikit-learn==1.6.1

import re, random
import numpy as np
import pandas as pd
from langdetect import detect, DetectorFactory

SEED = 42
random.seed(SEED); np.random.seed(SEED); DetectorFactory.seed = SEED

URL = "https://data.insideairbnb.com/turkey/marmara/istanbul/2026-09-25/data/listings.csv.gz"
df = pd.read_csv(URL, compression="gzip", low_memory=False)

print("1) Размер таблицы:", df.shape)

wanted = ["price", "description", "room_type", "property_type", "neighbourhood_cleansed",
          "accommodates", "bedrooms", "bathrooms", "review_scores_rating",
          "host_is_superhost", "amenities", "latitude", "longitude"]
print("\n2) Наличие колонок:")
for c in wanted:
    print(f"  {c:26s}", "ЕСТЬ" if c in df.columns else "НЕТ", "| пропусков:", df[c].isna().sum() if c in df.columns else "-")

print("\n   Другие колонки со словом price:", [c for c in df.columns if "price" in c.lower()])

# Формат и валюта цены
print("\n3) Тип колонки price:", df["price"].dtype)
print("   5 примеров цены (как в файле):", df["price"].dropna().sample(5, random_state=SEED).tolist())
raw = df["price"].dropna().astype(str)
print("   Символы, не являющиеся цифрой/точкой/запятой (валюта?):",
      raw.str.replace(r"[0-9.,]", "", regex=True).value_counts().head(5).to_dict())

price = pd.to_numeric(raw.str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
df["price_num"] = pd.to_numeric(df["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
print("   Статистика price_num:\n", df["price_num"].describe(percentiles=[.01, .5, .99]).round(1).to_string())

# Строки с ценой и описанием
has_price = df["price_num"].notna() & (df["price_num"] > 0)
has_desc = df["description"].notna() & (df["description"].astype(str).str.strip().str.len() > 20)
print("\n4) Строк с ценой (>0):", int(has_price.sum()))
print("   Строк с описанием (>20 симв.):", int(has_desc.sum()))
print("   Строк с обоими:", int((has_price & has_desc).sum()), "  <- нужно >= 3000")

# Язык описаний (выборка до 2000)
sub = df.loc[has_price & has_desc, "description"].astype(str)
sub = sub.sample(min(2000, len(sub)), random_state=SEED)
def clean(t): return re.sub(r"<[^>]+>", " ", t)
def lang(t):
    try: return detect(clean(t))
    except Exception: return "err"
langs = sub.map(lang)
print("\n5) Языки описаний (выборка", len(sub), "):")
print((langs.value_counts(normalize=True).head(8) * 100).round(1).to_string())
tr_chars = sub.str.contains(r"[çğışöüÇĞİŞÖÜ]").mean()
print("   Доля описаний с турецкими буквами (грубая оценка): %.1f%%" % (tr_chars * 100))

# Целевые метки для NLP
print("\n6) room_type:\n", df["room_type"].value_counts(dropna=False).to_string())
print("\n   property_type (топ-10):\n", df["property_type"].value_counts(dropna=False).head(10).to_string())
print("\n   Районов (neighbourhood_cleansed):", df["neighbourhood_cleansed"].nunique())
print("\n   Длина описаний, символов (медиана/90%):",
      sub.str.len().median(), sub.str.len().quantile(.9))

1) Размер таблицы: (27232, 90)

2) Наличие колонок:
  price                      ЕСТЬ | пропусков: 2679
  description                ЕСТЬ | пропусков: 162
  room_type                  ЕСТЬ | пропусков: 0
  property_type              ЕСТЬ | пропусков: 0
  neighbourhood_cleansed     ЕСТЬ | пропусков: 0
  accommodates               ЕСТЬ | пропусков: 0
  bedrooms                   ЕСТЬ | пропусков: 23
  bathrooms                  ЕСТЬ | пропусков: 27232
  review_scores_rating       ЕСТЬ | пропусков: 9661
  host_is_superhost          ЕСТЬ | пропусков: 0
  amenities                  ЕСТЬ | пропусков: 0
  latitude                   ЕСТЬ | пропусков: 0
  longitude                  ЕСТЬ | пропусков: 0

   Другие колонки со словом price: ['price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw']

3) Тип колонки price: object
   5 примеров цены (как в файле): ['$14,950.00', '$1,030.83', '$8,972.50', '$5,795.89', 

In [15]:
# Ячейка 2. Валюта и колонки price_quote_*
cols = ["price", "price_quote_price_per_night", "price_quote_total_price",
        "price_quote_checkin_date", "price_quote_checkout_date", "price_quote_raw"]
print(df[cols].dropna(subset=["price"]).head(8).to_string())
print("\nПропусков:", df[cols].isna().sum().to_dict())
print("\nbathrooms_text есть:", "bathrooms_text" in df.columns)
print("Колонки с currency:", [c for c in df.columns if "curr" in c.lower()])

        price  price_quote_price_per_night  price_quote_total_price price_quote_checkin_date price_quote_checkout_date                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [16]:
# Ячейка 3. Длина проживания и "месячные" цены
import json
d = df[df["price_num"].notna() & df["price_quote_raw"].notna()].copy()
d["nights"] = (pd.to_datetime(d["price_quote_checkout_date"]) - pd.to_datetime(d["price_quote_checkin_date"])).dt.days
def monthly(s):
    try: return json.loads(s)["quote"]["is_monthly_priced"]
    except Exception: return None
d["is_monthly"] = d["price_quote_raw"].map(monthly)
print("is_monthly_priced:\n", d["is_monthly"].value_counts(dropna=False).to_string())
print("\nНочей (процентили 5/25/50/75/95):", d["nights"].quantile([.05,.25,.5,.75,.95]).tolist())
print("\nМедиана price_num (₺) по группам:")
print(d.groupby("is_monthly")["price_num"].describe(percentiles=[.5,.95]).round(0).to_string())
print("\nКороткие котировки (<=7 ночей):", int((d["nights"] <= 7).sum()))
print("Короткие котировки с описанием:", int(((d["nights"] <= 7) & d["description"].notna()).sum()))
print("\nПо месяцу заезда:\n", pd.to_datetime(d["price_quote_checkin_date"]).dt.to_period("M").value_counts().sort_index().to_string())

is_monthly_priced:
 is_monthly
False    16465
True      8088

Ночей (процентили 5/25/50/75/95): [1.0, 1.0, 2.0, 100.0, 100.0]

Медиана price_num (₺) по группам:
              count    mean      std    min     50%      95%        max
is_monthly                                                             
False       16465.0  6339.0  10337.0  320.0  4490.0  15048.0   502353.0
True         8088.0  5378.0  64971.0   79.0  2106.0  10015.0  4989148.0

Короткие котировки (<=7 ночей): 16280
Короткие котировки с описанием: 16280

По месяцу заезда:
 price_quote_checkin_date
2026-09    15707
2026-10     6912
2026-11      546
2026-12      265
2027-01      626
2027-02      158
2027-03       96
2027-04       62
2027-05       50
2027-06       69
2027-07       16
2027-08       14
2027-09       32
Freq: M


In [17]:
# Ячейка 4. Очистка данных (запускается после ячеек 1 и 3)
import json, re
import numpy as np, pandas as pd

SEED = 42; np.random.seed(SEED)

# если ноутбук перезапускали, заново читаем данные
if "df" not in globals():
    URL = "https://data.insideairbnb.com/turkey/marmara/istanbul/2026-09-25/data/listings.csv.gz"
    df = pd.read_csv(URL, compression="gzip", low_memory=False)

d = df.copy()
print("Всего строк:", len(d))

# 1) Цена в лирах
d["price_try"] = pd.to_numeric(d["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
d = d[d["price_try"].notna() & (d["price_try"] > 0)]
print("С ценой > 0:", len(d))

# 2) Короткие котировки: не "месячные" и не больше 7 ночей
def is_monthly(s):
    try: return json.loads(s)["quote"]["is_monthly_priced"]
    except Exception: return None
d["is_monthly"] = d["price_quote_raw"].map(is_monthly)
d["nights"] = (pd.to_datetime(d["price_quote_checkout_date"]) - pd.to_datetime(d["price_quote_checkin_date"])).dt.days
d = d[(d["is_monthly"] == False) & (d["nights"] <= 7)]
print("Короткие котировки (не monthly, <=7 ночей):", len(d))

# 3) Описания: убираем HTML и маскируем контакты ДО любой работы с текстом
d = d[d["description"].notna()].copy()
cnt = {"email": 0, "url": 0, "phone": 0, "handle": 0}
def mask(t):
    t = re.sub(r"<[^>]+>", " ", str(t))
    t, n = re.subn(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]", t); cnt["email"] += n
    t, n = re.subn(r"(?:https?://|www\.)\S+", "[URL]", t); cnt["url"] += n
    t, n = re.subn(r"(?<!\w)@\w{3,}", "[HANDLE]", t); cnt["handle"] += n
    t, n = re.subn(r"(?<![\w])\+?\d[\d\s().-]{7,}\d(?![\w])", "[PHONE]", t); cnt["phone"] += n
    return re.sub(r"\s+", " ", t).strip()
d["description_clean"] = d["description"].map(mask)
d = d[d["description_clean"].str.len() > 20]
print("Замаскировано (количество замен):", cnt)
print("После фильтра описаний:", len(d))

# 4) Дубликаты по id
print("Дубликатов id:", int(d["id"].duplicated().sum()))
d = d.drop_duplicates("id")

# 5) Оставляем нужные колонки. price_quote_* и сырой price в признаки не идут.
keep = ["id", "price_try", "description_clean", "room_type", "property_type",
        "neighbourhood_cleansed", "accommodates", "bedrooms", "bathrooms_text",
        "review_scores_rating", "host_is_superhost", "amenities", "latitude", "longitude"]
d = d[keep].reset_index(drop=True)

print("\nИтого строк:", len(d))
print("Цена, ₺ (мин / 1% / медиана / 99% / макс):",
      d["price_try"].quantile([0, .01, .5, .99, 1]).round(0).tolist())
print("Пропуски по колонкам:\n", d.isna().sum().to_string())
print("\nroom_type:\n", d["room_type"].value_counts().to_string())

d.to_parquet("/content/clean_listings.parquet", index=False)
print("\nСохранено в /content/clean_listings.parquet (только Colab, в GitHub не кладём)")

Всего строк: 27232
С ценой > 0: 24553
Короткие котировки (не monthly, <=7 ночей): 16280
Замаскировано (количество замен): {'email': 0, 'url': 12, 'phone': 68, 'handle': 24}
После фильтра описаний: 16236
Дубликатов id: 0

Итого строк: 16236
Цена, ₺ (мин / 1% / медиана / 99% / макс): [320.0, 1465.0, 4500.0, 39366.0, 502353.0]
Пропуски по колонкам:
 id                           0
price_try                    0
description_clean            0
room_type                    0
property_type                0
neighbourhood_cleansed       0
accommodates                 0
bedrooms                     5
bathrooms_text              64
review_scores_rating      3505
host_is_superhost            0
amenities                    0
latitude                     0
longitude                    0

room_type:
 room_type
Entire home/apt    11598
Private room        4266
Hotel room           246
Shared room          126

Сохранено в /content/clean_listings.parquet (только Colab, в GitHub не кладём)


In [18]:
# Ячейка 5. Признаки, разбиение 80/20, baseline, Ridge, градиентный бустинг
import json, re
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, median_absolute_error

SEED = 42; np.random.seed(SEED)
d = pd.read_parquet("/content/clean_listings.parquet")

# 1) Выбросы: убираем верхний 1% цен (решение запишем в карточку датасета)
cap = d["price_try"].quantile(0.99)
d = d[d["price_try"] <= cap].reset_index(drop=True)
print("Порог цены (99%):", round(cap), "₺ | строк после:", len(d))

# 2) Разбиение 80/20
train, test = train_test_split(d, test_size=0.2, random_state=SEED)
train, test = train.copy(), test.copy()

# 3) Признаки
def parse_baths(t):
    if pd.isna(t): return np.nan
    m = re.search(r"(\d+(?:\.\d+)?)", str(t))
    if m: return float(m.group(1))
    return 0.5 if "half" in str(t).lower() else np.nan

def amenities_list(s):
    try: return json.loads(s)
    except Exception: return []

for part in (train, test):
    part["baths"] = part["bathrooms_text"].map(parse_baths)
    part["shared_bath"] = part["bathrooms_text"].astype(str).str.lower().str.contains("shared").astype(int)
    part["superhost"] = (part["host_is_superhost"] == "t").astype(int)
    part["no_rating"] = part["review_scores_rating"].isna().astype(int)
    part["am_list"] = part["amenities"].map(amenities_list)
    part["n_amenities"] = part["am_list"].map(len)

# 15 самых частых удобств по обучающей части
from collections import Counter
top_am = [a for a, _ in Counter(a for L in train["am_list"] for a in L).most_common(15)]
for part in (train, test):
    for i, a in enumerate(top_am):
        part[f"am_{i}"] = part["am_list"].map(lambda L, a=a: int(a in L))
am_cols = [f"am_{i}" for i in range(len(top_am))]

# редкие типы жилья -> Other (по обучающей части)
freq = train["property_type"].value_counts()
common = set(freq[freq >= 50].index)
for part in (train, test):
    part["ptype"] = part["property_type"].where(part["property_type"].isin(common), "Other")

cat = ["room_type", "ptype", "neighbourhood_cleansed"]
num = ["accommodates", "bedrooms", "baths", "review_scores_rating", "latitude", "longitude",
       "n_amenities", "shared_bath", "superhost", "no_rating"] + am_cols

# заполнение пропусков медианами обучающей части
med = train[num].median()
for part in (train, test):
    part[num] = part[num].fillna(med)

ytr, yte = train["price_try"].values, test["price_try"].values
ytr_log = np.log1p(ytr)

# 4) Baseline: медиана по району (запасной вариант: общая медиана)
dist_med = train.groupby("neighbourhood_cleansed")["price_try"].median()
glob_med = train["price_try"].median()
pred_base = test["neighbourhood_cleansed"].map(dist_med).fillna(glob_med).values

# 5) Ridge на логарифме цены
pre_lin = ColumnTransformer([
    ("c", OneHotEncoder(handle_unknown="ignore"), cat),
    ("n", StandardScaler(), num)])
ridge = Pipeline([("p", pre_lin), ("m", Ridge(alpha=1.0, random_state=SEED))])
ridge.fit(train[cat + num], ytr_log)
pred_ridge = np.expm1(ridge.predict(test[cat + num]))

# 6) Градиентный бустинг
pre_gb = ColumnTransformer([
    ("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                         encoded_missing_value=np.nan), cat),
    ("n", "passthrough", num)])
n_cat = len(cat)
gb = Pipeline([("p", pre_gb),
               ("m", HistGradientBoostingRegressor(
                   categorical_features=[True]*n_cat + [False]*len(num),
                   learning_rate=0.08, max_iter=400, random_state=SEED))])
gb.fit(train[cat + num], ytr_log)
pred_gb = np.expm1(gb.predict(test[cat + num]))

# 7) Результаты (в лирах) на отложенной выборке
def rep(name, p):
    print(f"{name:22s} MAE = {mean_absolute_error(yte, p):8.0f} ₺ | медианная ошибка = {median_absolute_error(yte, p):8.0f} ₺")
print("\nTrain:", len(train), "| Test:", len(test), "| медиана цены test:", round(np.median(yte)), "₺")
rep("Baseline (район)", pred_base)
rep("Ridge (log-цена)", pred_ridge)
rep("Gradient boosting", pred_gb)

# сохраняем артефакты в Colab (в GitHub пока не кладём)
import joblib
joblib.dump({"ridge": ridge, "gb": gb, "dist_med": dist_med, "glob_med": glob_med,
             "med": med, "top_am": top_am, "common_ptypes": sorted(common), "cap": cap,
             "cat": cat, "num": num}, "/content/ml_artifacts.joblib")
print("\nСохранено: /content/ml_artifacts.joblib")

Порог цены (99%): 39366 ₺ | строк после: 16073

Train: 12858 | Test: 3215 | медиана цены test: 4500 ₺
Baseline (район)       MAE =     2594 ₺ | медианная ошибка =     1600 ₺
Ridge (log-цена)       MAE =     1956 ₺ | медианная ошибка =     1112 ₺
Gradient boosting      MAE =     1412 ₺ | медианная ошибка =      744 ₺

Сохранено: /content/ml_artifacts.joblib


In [19]:
# Ячейка 6. Диапазон цены: калибровка по кросс-валидации, проверка на тесте
import joblib
from sklearn.model_selection import KFold, cross_val_predict

X_tr = train[cat + num]
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_log = cross_val_predict(gb, X_tr, ytr_log, cv=kf)
oof = np.expm1(oof_log)
ratio = ytr / oof                      # во сколько раз реальная цена отличалась от прогноза
q10, q90 = np.quantile(ratio, [0.10, 0.90])
print(f"Множители диапазона (10%..90%): {q10:.2f} .. {q90:.2f}")

lo, hi = pred_gb * q10, pred_gb * q90
cover = float(np.mean((yte >= lo) & (yte <= hi)))
print(f"Доля тестовых цен внутри диапазона: {cover*100:.1f}% (цель около 80%)")
print("Ширина диапазона, медиана:", round(float(np.median(hi - lo))), "₺")

# ошибка по ценовым группам (где модель слабее)
bins = pd.qcut(pd.Series(yte), 4, labels=["Q1 дешёвые", "Q2", "Q3", "Q4 дорогие"])
err = pd.DataFrame({"bin": bins, "ae": np.abs(yte - pred_gb), "price": yte})
print("\nMAE по квартилям цены:\n", err.groupby("bin", observed=True).agg(MAE=("ae", "mean"), median_price=("price", "median")).round(0).to_string())

art = joblib.load("/content/ml_artifacts.joblib")
art.update({"range_q10": float(q10), "range_q90": float(q90)})
joblib.dump(art, "/content/ml_artifacts.joblib")
print("\nДиапазон добавлен в /content/ml_artifacts.joblib")

Множители диапазона (10%..90%): 0.69 .. 1.48
Доля тестовых цен внутри диапазона: 82.1% (цель около 80%)
Ширина диапазона, медиана: 3612 ₺

MAE по квартилям цены:
                MAE  median_price
bin                             
Q1 дешёвые   676.0        2449.0
Q2           750.0        3723.0
Q3          1023.0        5543.0
Q4 дорогие  3202.0        9586.0

Диапазон добавлен в /content/ml_artifacts.joblib


In [20]:
# Ячейка 7. NLP: room_type по описанию, F1 по классам, порог «не уверен»
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

def make_clf():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000,
                                  sublinear_tf=True, lowercase=True)),
        ("lr", LogisticRegression(max_iter=2000, C=3.0, class_weight="balanced", random_state=SEED))])

Xtr, ytr_c = train["description_clean"], train["room_type"]
Xte, yte_c = test["description_clean"], test["room_type"]
classes = sorted(ytr_c.unique())
print("Классы:", classes)

# 1) Порог «не уверен» подбираем на валидации внутри train
Xa, Xv, ya, yv = train_test_split(Xtr, ytr_c, test_size=0.2, stratify=ytr_c, random_state=SEED)
clf_v = make_clf().fit(Xa, ya)
pv = clf_v.predict_proba(Xv); pred_v = clf_v.classes_[pv.argmax(1)]; conf_v = pv.max(1)

print("\nВалидация: порог -> доля ответов, macro-F1 на отвеченных")
best_t, best_f1 = None, -1
for t in np.arange(0.40, 0.96, 0.05):
    m = conf_v >= t
    cov = m.mean()
    f1m = f1_score(yv[m], pred_v[m], average="macro", labels=classes, zero_division=0) if m.sum() else 0
    print(f"  t={t:.2f}  ответов {cov*100:5.1f}%  macro-F1 {f1m:.3f}")
    if cov >= 0.80 and f1m > best_f1:
        best_t, best_f1 = float(round(t, 2)), f1m
if best_t is None: best_t = 0.50
print("Выбранный порог:", best_t, "(макс. macro-F1 при охвате >= 80%)")

# 2) Финальная модель на всём train, оценка на test
clf = make_clf().fit(Xtr, ytr_c)
pt = clf.predict_proba(Xte); pred_t = clf.classes_[pt.argmax(1)]; conf_t = pt.max(1)

print("\n=== TEST, все ответы (без порога) ===")
print(classification_report(yte_c, pred_t, digits=3, zero_division=0))

m = conf_t >= best_t
print(f"=== TEST, только уверенные (порог {best_t}) — отвечено {m.mean()*100:.1f}% ===")
print(classification_report(yte_c[m], pred_t[m], digits=3, zero_division=0))
print("«Не уверен» по истинным классам:\n", yte_c[~m].value_counts().to_string())
print("\nЧисло примеров в тесте по классам:\n", yte_c.value_counts().to_string())

art2 = {"clf": clf, "threshold": best_t, "classes": list(clf.classes_)}
joblib.dump(art2, "/content/nlp_artifacts.joblib")
print("\nСохранено: /content/nlp_artifacts.joblib")


Классы: ['Entire home/apt', 'Hotel room', 'Private room', 'Shared room']

Валидация: порог -> доля ответов, macro-F1 на отвеченных
  t=0.40  ответов  99.8%  macro-F1 0.662
  t=0.45  ответов  98.7%  macro-F1 0.664
  t=0.50  ответов  95.4%  macro-F1 0.676
  t=0.55  ответов  89.6%  macro-F1 0.693
  t=0.60  ответов  85.5%  macro-F1 0.717
  t=0.65  ответов  81.7%  macro-F1 0.740
  t=0.70  ответов  75.5%  macro-F1 0.765
  t=0.75  ответов  71.3%  macro-F1 0.784
  t=0.80  ответов  65.1%  macro-F1 0.801
  t=0.85  ответов  54.2%  macro-F1 0.813
  t=0.90  ответов  39.7%  macro-F1 0.858
  t=0.95  ответов  19.2%  macro-F1 0.902
Выбранный порог: 0.65 (макс. macro-F1 при охвате >= 80%)

=== TEST, все ответы (без порога) ===
                 precision    recall  f1-score   support

Entire home/apt      0.956     0.903     0.929      2332
     Hotel room      0.923     0.878     0.900        41
   Private room      0.816     0.779     0.797       816
    Shared room      0.052     0.385     0.091      

In [21]:
# Ячейка 8. Сравнение вариантов меток и весов классов
MAP4 = {c: c for c in ["Entire home/apt", "Hotel room", "Private room", "Shared room"]}
MAP3 = {"Entire home/apt": "Entire home/apt", "Hotel room": "Hotel room",
        "Private room": "Private/Shared room", "Shared room": "Private/Shared room"}

def make_clf(cw):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000,
                                  sublinear_tf=True, lowercase=True)),
        ("lr", LogisticRegression(max_iter=2000, C=3.0, class_weight=cw, random_state=SEED))])

def run_val(name, ymap, cw):
    ytr_ = train["room_type"].map(ymap)
    Xa, Xv, ya, yv = train_test_split(Xtr, ytr_, test_size=0.2, stratify=ytr_, random_state=SEED)
    c = make_clf(cw).fit(Xa, ya)
    p = c.predict_proba(Xv); pred = c.classes_[p.argmax(1)]; conf = p.max(1)
    f1_all = f1_score(yv, pred, average="macro")
    best_t, best_f = 0.5, -1
    for t in np.arange(0.40, 0.96, 0.05):
        m = conf >= t
        if m.mean() >= 0.80 and m.sum():
            f = f1_score(yv[m], pred[m], average="macro", zero_division=0)
            if f > best_f: best_t, best_f = float(round(t, 2)), f
    return dict(name=name, ymap=ymap, cw=cw, f1_val=f1_all, t=best_t, f1_val_t=best_f)

variants = [run_val("A: 4 класса, без весов", MAP4, None),
            run_val("B: 3 класса, с весами", MAP3, "balanced"),
            run_val("C: 3 класса, без весов", MAP3, None)]
print("Валидация (macro-F1 без порога | порог | macro-F1 при охвате>=80%):")
for v in variants:
    print(f"  {v['name']:26s} {v['f1_val']:.3f} | t={v['t']} | {v['f1_val_t']:.3f}")

best = max(variants, key=lambda v: v["f1_val"])
print("\nВыбран по валидации:", best["name"])

# финальная модель и ОДНА оценка на тесте
ymap = best["ymap"]
ytr_f, yte_f = train["room_type"].map(ymap), test["room_type"].map(ymap)
clf = make_clf(best["cw"]).fit(Xtr, ytr_f)
pt = clf.predict_proba(Xte); pred_t = clf.classes_[pt.argmax(1)]; conf_t = pt.max(1)
print("\n=== TEST, все ответы ===")
print(classification_report(yte_f, pred_t, digits=3, zero_division=0))
m = conf_t >= best["t"]
print(f"=== TEST, уверенные (порог {best['t']}), отвечено {m.mean()*100:.1f}% ===")
print(classification_report(yte_f[m], pred_t[m], digits=3, zero_division=0))
print("Примеров в тесте:\n", yte_f.value_counts().to_string())

joblib.dump({"clf": clf, "threshold": best["t"], "classes": list(clf.classes_),
             "label_map": ymap, "variant": best["name"]}, "/content/nlp_artifacts.joblib")
print("\nСохранено: /content/nlp_artifacts.joblib")

Валидация (macro-F1 без порога | порог | macro-F1 при охвате>=80%):
  A: 4 класса, без весов     0.678 | t=0.8 | 0.718
  B: 3 класса, с весами      0.892 | t=0.7 | 0.941
  C: 3 класса, без весов     0.880 | t=0.8 | 0.966

Выбран по валидации: B: 3 класса, с весами

=== TEST, все ответы ===
                     precision    recall  f1-score   support

    Entire home/apt      0.949     0.942     0.946      2332
         Hotel room      0.923     0.878     0.900        41
Private/Shared room      0.842     0.861     0.851       842

           accuracy                          0.920      3215
          macro avg      0.905     0.894     0.899      3215
       weighted avg      0.921     0.920     0.920      3215

=== TEST, уверенные (порог 0.7), отвечено 86.8% ===
                     precision    recall  f1-score   support

    Entire home/apt      0.969     0.975     0.972      2036
         Hotel room      0.947     0.900     0.923        40
Private/Shared room      0.926     0.913   

In [22]:
# Ячейка 9. Как ошибки NLP влияют на цену (сквозная проверка NLP -> ML)
t = best["t"]
nlp_pred = pd.Series(pred_t, index=test.index)
nlp_conf = pd.Series(conf_t, index=test.index)

map_to_ml = {"Entire home/apt": "Entire home/apt", "Hotel room": "Hotel room",
             "Private/Shared room": "Private room"}
rt_nlp = nlp_pred.map(map_to_ml)

def mae_with(room_series, label):
    X = test[cat + num].copy()
    X["room_type"] = room_series.values
    p = np.expm1(gb.predict(X))
    print(f"{label:48s} MAE = {mean_absolute_error(yte, p):6.0f} ₺ | медианная = {median_absolute_error(yte, p):5.0f} ₺")

print("ML (градиентный бустинг), MAE на тесте:")
mae_with(test["room_type"], "1) настоящий room_type")
mae_with(rt_nlp, "2) room_type от NLP (все ответы)")
rt_conf = rt_nlp.where(nlp_conf >= t)          # при «не уверен» -> пропуск
mae_with(rt_conf, "3) от NLP, при «не уверен» -> пропуск")
print(f"\nДоля «не уверен»: {(nlp_conf < t).mean()*100:.1f}%")

# Метрики для README (реальные числа из этого запуска)
import json
report = {"ml": {"split": "80/20, seed 42", "price_cap_try": float(cap),
                 "n_train": len(train), "n_test": len(test),
                 "mae_baseline": float(mean_absolute_error(yte, pred_base)),
                 "mae_ridge": float(mean_absolute_error(yte, pred_ridge)),
                 "mae_gb": float(mean_absolute_error(yte, pred_gb))},
          "nlp": {"variant": best["name"], "threshold": t}}
json.dump(report, open("/content/metrics.json", "w"), ensure_ascii=False, indent=2)

import os
for f in ["ml_artifacts.joblib", "nlp_artifacts.joblib", "metrics.json"]:
    print(f"{f}: {os.path.getsize('/content/'+f)/1e6:.2f} МБ")

ML (градиентный бустинг), MAE на тесте:
1) настоящий room_type                           MAE =   1412 ₺ | медианная =   744 ₺
2) room_type от NLP (все ответы)                 MAE =   1416 ₺ | медианная =   747 ₺
3) от NLP, при «не уверен» -> пропуск            MAE =   1415 ₺ | медианная =   752 ₺

Доля «не уверен»: 13.2%
ml_artifacts.joblib: 1.60 МБ
nlp_artifacts.joblib: 2.28 МБ
metrics.json: 0.00 МБ


In [23]:
# Ячейка 10. (а) проверка словаря TF-IDF, (б) вклад room_type при разных наборах признаков
vec = clf.named_steps["tfidf"]
vocab = vec.get_feature_names_out()
sus = [v for v in vocab if re.search(r"\d{6,}", v)]
print("Размер словаря:", len(vocab), "| токенов с 6+ цифрами подряд:", len(sus))
print("Примеры (только токены):", sus[:10])

def fit_gb(cat_cols):
    pre = ColumnTransformer([
        ("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                             encoded_missing_value=np.nan), cat_cols),
        ("n", "passthrough", num)])
    m = Pipeline([("p", pre), ("m", HistGradientBoostingRegressor(
        categorical_features=[True]*len(cat_cols) + [False]*len(num),
        learning_rate=0.08, max_iter=400, random_state=SEED))])
    return m.fit(train[cat_cols + num], ytr_log)

def mae_cols(model, cat_cols, room_series=None):
    X = test[cat_cols + num].copy()
    if room_series is not None: X["room_type"] = room_series.values
    p = np.expm1(model.predict(X))
    return mean_absolute_error(yte, p), median_absolute_error(yte, p)

print("\nБез property_type (признаки: room_type + район + числовые):")
c_np = ["room_type", "neighbourhood_cleansed"]
g_np = fit_gb(c_np)
for lab, ser in [("настоящий room_type", test["room_type"]),
                 ("room_type от NLP", rt_nlp),
                 ("от NLP, «не уверен» -> пропуск", rt_conf)]:
    a, b = mae_cols(g_np, c_np, ser)
    print(f"  {lab:34s} MAE = {a:6.0f} ₺ | медианная = {b:5.0f} ₺")

c_none = ["neighbourhood_cleansed"]
g_none = fit_gb(c_none)
a, b = mae_cols(g_none, c_none)
print(f"  {'совсем без room_type':34s} MAE = {a:6.0f} ₺ | медианная = {b:5.0f} ₺")

Размер словаря: 34853 | токенов с 6+ цифрами подряд: 0
Примеры (только токены): []

Без property_type (признаки: room_type + район + числовые):
  настоящий room_type                MAE =   1444 ₺ | медианная =   778 ₺
  room_type от NLP                   MAE =   1453 ₺ | медианная =   791 ₺
  от NLP, «не уверен» -> пропуск     MAE =   1454 ₺ | медианная =   791 ₺
  совсем без room_type               MAE =   1456 ₺ | медианная =   775 ₺


In [24]:
# Ячейка 11. Финальные артефакты для репозитория
import os, json, zipfile, joblib, sklearn
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import classification_report

c_np = ["room_type", "neighbourhood_cleansed"]

# диапазон для финальной модели (калибровка по кросс-валидации на train)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.expm1(cross_val_predict(g_np, train[c_np + num], ytr_log, cv=kf))
q10, q90 = (float(x) for x in np.quantile(ytr / oof, [0.10, 0.90]))

def cover_with(room_series):
    X = test[c_np + num].copy(); X["room_type"] = room_series.values
    p = np.expm1(g_np.predict(X))
    return float(np.mean((yte >= p*q10) & (yte <= p*q90))), float(mean_absolute_error(yte, p))

cov_true, mae_true = cover_with(test["room_type"])
cov_nlp, mae_nlp = cover_with(rt_conf)
print(f"Диапазон x{q10:.2f}..x{q90:.2f} | покрытие (истинный тип): {cov_true*100:.1f}% | (тип от NLP): {cov_nlp*100:.1f}%")

# F1 по классам NLP на тесте (все ответы и при пороге «не уверен»)
def f1_block(y_true, y_pred):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "f1_by_class": {k: round(v["f1-score"], 3) for k, v in rep.items()
                        if k not in ("accuracy", "macro avg", "weighted avg")},
        "support_by_class": {k: int(v["support"]) for k, v in rep.items()
                             if k not in ("accuracy", "macro avg", "weighted avg")},
        "accuracy": round(rep["accuracy"], 3),
        "macro_f1": round(rep["macro avg"]["f1-score"], 3),
    }

mask_conf = conf_t >= best["t"]
nlp_metrics = {
    "variant": best["name"], "threshold": best["t"],
    "n_test": int(len(yte_f)),
    "no_threshold": f1_block(yte_f, pred_t),
    "with_threshold": {**f1_block(yte_f[mask_conf], pred_t[mask_conf]),
                       "answered_share": round(float(mask_conf.mean()), 3)},
}
print("NLP F1 (без порога):", nlp_metrics["no_threshold"]["f1_by_class"],
      "| macro", nlp_metrics["no_threshold"]["macro_f1"])
print("NLP F1 (порог %.2f):" % best["t"], nlp_metrics["with_threshold"]["f1_by_class"],
      "| macro", nlp_metrics["with_threshold"]["macro_f1"],
      "| отвечено", nlp_metrics["with_threshold"]["answered_share"])

os.makedirs("/content/export/models", exist_ok=True)
joblib.dump({"model": g_np, "cat": c_np, "num": num, "median_fill": med,
             "top_amenities": top_am, "range_q10": q10, "range_q90": q90,
             "price_cap_try": float(cap), "district_median": dist_med,
             "global_median": float(glob_med)},
            "/content/export/models/price_model.joblib")
joblib.dump({"clf": clf, "threshold": best["t"], "label_map": best["ymap"],
             "classes": list(clf.classes_), "to_ml": map_to_ml},
            "/content/export/models/room_type_clf.joblib")

metrics = {
 "seed": 42, "sklearn_version": sklearn.__version__,
 "data": {"rows_clean": int(len(d)), "price_cap_try": float(cap),
          "n_train": int(len(train)), "n_test": int(len(test))},
 "ml": {"mae_baseline_district_median": float(mean_absolute_error(yte, pred_base)),
        "mae_ridge": float(mean_absolute_error(yte, pred_ridge)),
        "mae_gb_with_property_type": float(mean_absolute_error(yte, pred_gb)),
        "mae_gb_final_true_room_type": mae_true,
        "mae_gb_final_nlp_room_type": mae_nlp,
        "range_q10": q10, "range_q90": q90,
        "range_coverage_test_true": cov_true, "range_coverage_test_nlp": cov_nlp},
 "nlp": nlp_metrics}
json.dump(metrics, open("/content/export/models/metrics.json", "w"), ensure_ascii=False, indent=2)

with zipfile.ZipFile("/content/models_export.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir("/content/export/models"):
        z.write("/content/export/models/" + f, "models/" + f)
print("scikit-learn:", sklearn.__version__)
print("Размер архива, МБ:", round(os.path.getsize("/content/models_export.zip")/1e6, 2))
try:
    from google.colab import files; files.download("/content/models_export.zip")
except Exception as e:
    print("Скачай файл вручную из панели Files:", e)

Диапазон x0.69..x1.49 | покрытие (истинный тип): 81.3% | (тип от NLP): 80.9%
NLP F1 (без порога): {'Entire home/apt': 0.946, 'Hotel room': 0.9, 'Private/Shared room': 0.851} | macro 0.899
NLP F1 (порог 0.70): {'Entire home/apt': 0.972, 'Hotel room': 0.923, 'Private/Shared room': 0.92} | macro 0.938 | отвечено 0.868
scikit-learn: 1.6.1
Размер архива, МБ: 1.53


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
# Ячейка 12. Центры районов для приложения
cent = train.groupby("neighbourhood_cleansed")[["latitude", "longitude"]].median()
art_p = joblib.load("/content/export/models/price_model.joblib")
art_p["district_centroids"] = {k: (float(r.latitude), float(r.longitude)) for k, r in cent.iterrows()}
joblib.dump(art_p, "/content/export/models/price_model.joblib")

with zipfile.ZipFile("/content/models_export.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir("/content/export/models"):
        z.write("/content/export/models/" + f, "models/" + f)
print("Районов:", len(cent))
try:
    from google.colab import files; files.download("/content/models_export.zip")
except Exception as e:
    print("Скачай файл вручную:", e)

Районов: 37


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# Ячейка 13. Кросс-валидация MAE (5 фолдов, seed 42)
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cols = c_np + num   # признаки финальной модели (c_np задан в ячейке 11)
X, y = train[cols], ytr   # ytr — массив цен train (numpy)

maes_model, maes_base = [], []
for tr, va in kf.split(X):
    m = clone(g_np).fit(X.iloc[tr], np.log1p(y[tr]))
    pred = np.expm1(m.predict(X.iloc[va]))
    maes_model.append(mean_absolute_error(y[va], pred))
    med = train.iloc[tr].groupby("neighbourhood_cleansed")["price_try"].median()
    base = train.iloc[va]["neighbourhood_cleansed"].map(med).fillna(np.median(y[tr]))
    maes_base.append(mean_absolute_error(y[va], base))

for name, v in [("Бустинг", maes_model), ("Baseline: медиана по району", maes_base)]:
    print(f"{name:30s} MAE = {np.mean(v):7.0f} ± {np.std(v):4.0f} ₺ (по 5 фолдам: {np.round(v)})")

Бустинг                        MAE =    1478 ±   26 ₺ (по 5 фолдам: [1459. 1515. 1456. 1504. 1453.])
Baseline: медиана по району    MAE =    2585 ±   45 ₺ (по 5 фолдам: [2593. 2665. 2574. 2531. 2561.])
